# Library AI Chatbot

**Author:** Adriel Agustinus Jie  
**Assessment:** Option 1 — RAG Implementation  
**Repository:** [adrieljie/RAG-Implementation](https://github.com/adrieljie/RAG-Implementation)

## Completed Work

### Problem Statement

Readers may want to understand more about the characters, events, and motivations in a book, but they may not always have someone available to discuss it with. Finding passages that answer their questions can also require searching through different parts of the book.

This project builds a RAG-based book assistant that allows readers to ask questions about a collection of books. It retrieves relevant passages and generates answers with source references so readers can check the supporting evidence.

### Project Files

```RAG-Implementation/
├── README.md
├── requirements.txt
├── RAG_baseline.ipynb
├── RAG_experiment.ipynb
├── review.ipynb
└── docs/
    ├── alice_in_wonderland.txt
    ├── frankenstein.txt
    ├── pride_and_prejudice.txt
    └── sherlock_holmes.txt
```

- [RAG_baseline.ipynb](RAG_baseline.ipynb): baseline implementation and saved evaluation outputs.
- [RAG_experiment.ipynb](RAG_experiment.ipynb): experiment implementation, where last saved code is similarity score threshold with `0.7` threshold score.
- [review.ipynb](review.ipynb): saved examples, manual reviews, and experiment notes.
- [requirements.txt](requirements.txt): direct dependencies with pinned versions.

The local environment, downloaded models, and generated vector database are not required in the repository. The database is created through the Load Data menu.

### Dataset & Preprocessing

The dataset contains four English-language books from Project Gutenberg, stored as UTF-8 `.txt` files.

| Book | Author | Filename | File size |
|---|---|---|---:|
| [Alice's Adventures in Wonderland](https://www.gutenberg.org/ebooks/11) | Lewis Carroll | `alice_in_wonderland.txt` | 174,311 bytes |
| [Frankenstein](https://www.gutenberg.org/ebooks/84) | Mary Wollstonecraft Shelley | `frankenstein.txt` | 448,885 bytes |
| [Pride and Prejudice](https://www.gutenberg.org/ebooks/1342) | Jane Austen | `pride_and_prejudice.txt` | 772,386 bytes |
| [The Adventures of Sherlock Holmes](https://www.gutenberg.org/ebooks/1661) | Arthur Conan Doyle | `sherlock_holmes.txt` | 607,606 bytes |

Total dataset size: 4 files, 2,003,188 bytes, approximately 2 MB.

The books contain different characters, events, and passages, making them suitable for testing retrieval across a collection. Plain text was selected to avoid PDF layout and OCR problems.

The original files retain Project Gutenberg's usage notices and license. Their notices explain the applicable reuse terms and that users outside the United States need to check local copyright rules. Cleaning removes boilerplate from the searchable text, not from the original files.

### Architecture

```text
INGESTION
TXT files → duplicate check → text loading and metadata extraction → preprocessing → chunks → BGE embeddings → Chroma

QUESTION ANSWERING
Question → BGE query embedding → top-5 similarity retrieval → full passages + question + prompt rules → Qwen → answer + retrieved source references
```

The embedding model represents questions and passages as vectors for retrieval. Qwen receives the retrieved text and generates the answer, it does not search Chroma directly.

Ingestion, Preprocessing, and Metadata

`load_document()` reads `.txt` files directly inside `DOCS_PATH`. It does not recursively load subfolders. `TextLoader` uses UTF-8 decoding.

Before preprocessing, the program extracts `title`, `author`, `release_date`, and `language` from the Gutenberg headers. Missing values become `"Unknown"`. It also preserves `source` and adds `hash_file`, a SHA-256 hash of the original file bytes.

The preprocessing function:

1. Keeps the content between the Gutenberg START and END markers when both match.
2. Removes lines containing only asterisks.
3. Removes standalone `[Illustration]` markers.
4. Reduces repeated blank lines and trims leading and trailing whitespace.

It does not remove every illustration caption, multiline illustration block, or table of contents.

`RecursiveCharacterTextSplitter` uses a maximum size of **1,000 characters**, **0 overlap**, and separators in this order: `"\n\n"`, `"\n"`, `" "`, `""`. These are character counts, not token counts.

Chunks inherit document metadata and receive a numeric `chunk_id`, starting at **0 within each source file**. A filename and chunk ID together identify a passage. Plain-text files do not provide reliable page numbers.

Duplicate prevention checks the file hash against stored metadata and against hashes already seen in the current loading operation. Identical file contents are skipped even if the filename differs.

### Retrieval, Generation, and Source References

The baseline uses similarity search with cosine distance, returning up to five chunks. No similarity threshold is applied in the baseline.

The complete retrieved chunks are joined and supplied with the question. The system prompt asks Qwen to use only the context, combine relevant passages, consider the order of events, remain concise but complete, and avoid inventing information. The user prompt also asks it to say it does not know when the documents do not provide an answer.

With the default settings, the retrieved context contains up to roughly 5,000 characters, plus separators. The question and instructions add more input. There is no separate token-based context trimming.

Generation uses `max_new_tokens=512`, `do_sample=False`, and `return_full_text=False`. The 512-token limit applies to the generated output, not the input.

If retrieval returns no passages, the function returns `"I couldn't find relevant information."` without calling the LLM. When passages are returned but are irrelevant, abstention depends on the model following the prompt.

The output prints the title, author, chunk ID, and a 250-character evidence preview for each retrieved passage. The LLM receives the full passage. These references describe the retrieved context, not verified citations attached to individual answer claims.

To inspect full evidence after initialization and ingestion:

```python
answer, retrieved_docs = ask_question(
    "Who does Sherlock Holmes introduce as his friend and associate to the veiled lady?"
)

for doc in retrieved_docs:
    print(doc.metadata["source"], doc.metadata["chunk_id"])
    print(doc.page_content)
```

### Tools and Models

Component | Model or tool | Reason for selection |
|---|---|---|
| LLM | `Qwen/Qwen2.5-1.5B-Instruct` | An instruction-tuned model suited to answering questions from a supplied context. Its approximately 1.5B parameters make it a more manageable starting point for a laptop than a 7B model at the same numerical precision. This does not imply better answer quality than larger models. |
| Embedding model | `BAAI/bge-base-en-v1.5` | Supports English passage retrieval, matching the collection and task. Its published retrieval benchmark score is higher than the small version, while its embeddings have fewer dimensions than the large version. It was selected as a balance between benchmark performance and vector storage requirements. |
| Vector store | ChromaDB | Stores embeddings, text, and metadata locally, supports metadata filtering for duplicate checks, and preserves the collection between sessions without a separate database server. |
| Framework | LangChain integrations | Connect loading, chunking, embedding, retrieval, and generation with compatible components, reducing the code needed to connect them manually. The trade-off is additional dependencies and version compatibility requirements. |
| Text splitter | `RecursiveCharacterTextSplitter` | Tries natural text boundaries before smaller splits and does not require consistent chapter headings. Unlike cutting at fixed character positions, it tries to preserve paragraph and word boundaries, although complete ideas can still cross chunks. |
| Retrieval method | Similarity search with cosine distance, `k=5` | Provides a simple, inspectable baseline for semantic retrieval. It limits context to five passages without first tuning a score cutoff. It can still return irrelevant passages when the collection has no answer. |

The choices prioritize a manageable local baseline. They have not all been compared experimentally with alternatives, so they are not presented as the best-performing options.

Technical references: [Qwen model card](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [BGE model card and benchmarks](https://huggingface.co/BAAI/bge-base-en-v1.5), [Chroma integration](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma), and [recursive splitter documentation](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter).

### Configuration

``` DOCS_PATH = "./docs/"
DB_PATH = "./db/ChromaDB/"
COLLECTION_NAME = "books_dedup"

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
DEVICE = ""

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 0
K = 5
MAX_NEW_TOKENS = 512
```
Setting | Purpose |
|---|---|
| `DOCS_PATH` | Folder containing the `.txt` books. |
| `DB_PATH` / `COLLECTION_NAME` | Location and collection used by Chroma. |
| `EMBEDDING_MODEL` | Model used to embed documents and questions. |
| `LLM_MODEL` | Model used to generate answers. |
| `CHUNK_SIZE` / `CHUNK_OVERLAP` | Intended chunking settings; see the implementation note below. |
| `DEVICE` | Device used to run the LLM. |
| `MAX_NEW_TOKENS` | Maximum generated answer length in tokens. |
| `K` | Maximum number of retrieved passages. |

In the development environment, `DEVICE=0` resolved to `mps:0`, using the Apple GPU. Device selection can differ between systems. Use `"mps"` explicitly for an available Apple MPS device, or `-1` for CPU execution. Check the loaded LLM device with:

```python
print(pipeline.pipeline.device)
```

This device setting applies to the LLM; it is not explicitly passed to the embedding model.

The provider is local Hugging Face model execution. No paid API, API key, or `.env` file is required. The experiment notebook has separate settings; baseline configuration changes do not automatically update it.

After changing configuration, rerun the affected cells. Changing the embedding model requires rebuilding the embeddings in a fresh collection. Changing preprocessing or chunking also requires rebuilding the index and checking the reference chunk IDs used for evaluation.

### Setup and Run Commands

The inspected environment uses Python 3.10.0 on an Apple Silicon Mac. LLM execution was checked on MPS (`mps:0`). This setup does not require an NVIDIA GPU.

Allow several GB of disk space and free memory for model downloads and execution. An exact minimum RAM requirement has not been measured. CPU execution is configurable but may be slower.

Internet access is needed for dependency installation and initial model downloads. Downloaded models are cached locally.

**1. Clone the repository and create an environment** (macOS/Linux):

```bash
git clone https://github.com/adrieljie/RAG-Implementation.git
cd RAG-Implementation
python3 -m venv .venv
source .venv/bin/activate
```

**2. Install dependencies:**

```bash
python -m pip install -r requirements.txt
```

The dependency file pins versions, including `torch==2.14.1`, `transformers==5.20.1`, `sentence-transformers==6.1.0`, `chromadb==1.5.9`, and `ragas==0.3.9`. The complete list is in [requirements.txt](requirements.txt). The `langchain-community==0.4.1` pin retains a module needed by the installed Ragas version.

**3. Run the CLI**

From the repository directory, run:

`python RAG.py`

The application displays a text menu:

```
Library AI Chatbot!

Menu:
1. Load Data
2. Ask LLM
3. Exit

```

1. Select 1. Load Data to process the books and store their embeddings in Chroma.

2. Select 2. Ask LLM to enter questions about the books.

3. Type `quit` or `Quit` to return to the main menu.

4. Select 3. Exit to close the application.

On the first run, model loading may take longer because the models need to be downloaded. The database is stored locally and can be reused on later runs.

`RAG.py` runs the application only. It does not execute retrieval or generation evaluation.

### AI Assistance and References

#### What I Built Independently

I built and integrated the baseline pipeline, including document loading, text chunking, embedding generation, retrieval, and answer generation. I also ran the experiments and manually reviewed the system’s outputs.

#### AI Assistance

I used AI assistance to help implement duplicate checks and evaluation metrics, prepare documentation, and review and debug parts of the baseline code. AI also helped explain library usage, configuration options, and dependency errors.

#### What I Checked Myself

I inspected the loaded documents, metadata, processed text, and chunks. I compared retrieved passages and generated answers with the source books to distinguish retrieval failures from generation failures.

I tested similarity thresholds of 0.3, 0.5, and 0.7 and compared their retrieval results. I also tested follow-up question rewriting and identified cases where it changed the question’s meaning or introduced unsupported information.

I reviewed the suggested evaluation metrics and requested changes, including using only faithfulness and answer correctness for generation evaluation. I checked the reference chunk IDs and retrieved results when retrieval evaluation initially returned all-zero scores, and compared automated generation scores with my manual assessment of individual answers.

I asked for explanations of duplicate checks and evaluation metrics, and tested dependency and debugging fixes. When errors remained, I reported them for further investigation.

I reviewed the AI-drafted documentation for consistency with my implementation and results. I also started testing the setup from a GitHub clone with a new environment, a fresh database, and a separate model cache.

#### References

- [Qwen2.5-1.5B-Instruct model card](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)
- [BGE-base-en-v1.5 model card and benchmarks](https://huggingface.co/BAAI/bge-base-en-v1.5)
- [LangChain Chroma integration](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)
- [Recursive character splitting](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter)
- [Ragas documentation](https://docs.ragas.io/)

## Observed Results

### Baseline Evaluation Result

Five selected factual questions were evaluated: four about Alice and one about Sherlock Holmes. Each has a selected reference chunk. Generation evaluation uses the first reference answer listed for each question.

| Question | First reference rank | Hit@5 | Reciprocal rank |
|---|---:|---:|---:|
| Why does Alice follow the White Rabbit? | 3 | 1 | 0.333 |
| What opens the small door behind the curtain? | 1 | 1 | 1.000 |
| Why can't Alice climb the table leg to reach the key? | 1 | 1 | 1.000 |
| Who wakes Alice at the end of her adventure? | Not found | 0 | 0.000 |
| Who does Sherlock Holmes introduce as his friend and associate to the veiled lady? | 1 | 1 | 1.000 |
 
 ____

| Metric | Saved result | Interpretation |
|---|---:|---|
| Hit@5 | 80.00% | A reference chunk was retrieved for four of five questions. |
| MRR@5 | 0.667 | Mean reciprocal rank of the first reference chunk; a miss contributes zero. |
| Mean faithfulness | 0.667 | Automated assessment of support for answer claims in retrieved context. |
| Mean answer correctness | 0.656 | Automated assessment against reference answers, using factual comparison and semantic similarity. |

I selected Hit@5 and MRR@5 because the system retrieves up to five passages per question. Hit@5 checks whether at least one reference passage appears in those results, while MRR@5 checks how highly the first reference passage is ranked. Together, they help identify whether retrieval misses the expected evidence or ranks it below other passages. However, they do not measure the relevance of every retrieved chunk or the completeness of evidence for multi-passage questions.

I selected faithfulness and answer correctness because the system should produce answers that are both supported by the retrieved documents and correct compared with reference answers. Faithfulness checks whether the answer’s claims are supported by the retrieved context. Answer correctness evaluates the answer against the reference answer. These metrics complement each other, an answer may contain supported information but still fail to answer the question correctly or completely.

Generation evaluation uses the same local Qwen model as generation, so it is self-evaluation. These scores are not percentages of correctly answered questions.

### Saved Examples and Failure Analysis

Answers below reproduce saved model outputs. Evidence excerpts are shortened from the identified passages, with line breaks normalized for readability. Full passages can be inspected using `retrieved_docs`. A reference passage that was missed is explicitly distinguished from retrieved evidence.

#### Example 1 — Supported answer

**Question:** Who does Sherlock Holmes introduce as his friend and associate to the veiled lady?

**Generated answer:** Sherlock Holmes introduces Dr. Watson as his friend and associate to the veiled lady.

**Retrieved passages, in order:** `sherlock_holmes.txt` — chunks 425, 451, 587, 406, 368.

**Evidence from chunk 425:**

> This is my intimate friend and associate, Dr. Watson, before whom you can speak as freely as before myself.

**Review:** The answer identifies the correct person, supported directly by the first retrieved passage.

#### Example 2 — Relevant evidence retrieved, answer incomplete

**Question:** What opens the small door behind the curtain?

**Generated answer:** Alice opens the small door behind the curtain.

**Retrieved passages, in order:** `alice_in_wonderland.txt` — chunk 8; `sherlock_holmes.txt` — chunks 723, 52, 645; `alice_in_wonderland.txt` — chunk 9.

**Evidence from Alice, chunk 8:**

> she tried the little golden key in the lock, and to her great delight it fitted!

**Review:** The answer should identify the little golden key. The correct evidence was ranked first, but the answer names Alice instead of the object used. This is a generation failure despite a retrieval hit.

The automated answer-correctness score was approximately 0.900, illustrating why evaluator scores require manual checking.

#### Example 3 — Missing reference evidence and incorrect generation

**Question:** Who wakes Alice at the end of her adventure?

**Generated answer:** Alice doesn't wake anyone at the end of her adventure. The text describes Alice's journey through Wonderland, including meeting various characters such as the Mad Hatter, the March Hare, and the Dormouse. However, there is no mention of anyone waking Alice at the end of her adventure.

**Retrieved passages, in order:** `alice_in_wonderland.txt` — chunks 7, 51, 40, 96, 52.

**Excerpt from retrieved chunk 7:**

> Alice was not a bit hurt, and she jumped up on to her feet in a moment

**Review:** This retrieved passage concerns an earlier event. The answer also reverses the question from who wakes Alice to whether Alice wakes someone. The reference answer is her sister, supported by chunk 167, which was not retrieved*

> “Wake up, Alice dear!” said her sister

This example contains both a retrieval miss and a generation error. Chunk 167 is provided here for evaluation, not presented as evidence the model received.

### Optional Exploration: Similarity Threshold

#### What did I experiment with?
Adding a similarity score threshold to the retrieval method.

#### Why did I do this experiment?
In the baseline, the retriever still returned chunks for questions unrelated to the documents. Some answers were also incorrect because the retrieved chunks did not contain the information needed to answer the questions.

#### How did I conduct this experiment?
I added the `score_threshold` parameter to the retrieval step. I tested thresholds of 0.7, 0.5, and 0.3.

#### What did I find?

| Configuration | Hit@5 | MRR@5 | Mean faithfulness | Mean answer correctness |
|---|---:|---:|---:|---:|
| Baseline, no threshold | 80.00% | 0.667 | 0.667 | 0.656 |
| Threshold 0.7 | 20.00% | 0.067 | 0.200 | 0.193 |
| Threshold 0.5 | 80.00% | 0.667 | 0.667 | 0.656 |
| Threshold 0.3 | 80.00% | 0.667 | 0.667 | 0.656 |

In the recorded five-question evaluation, `0.7` removed enough evidence to reduce retrieval results. Thresholds `0.5` and `0.3` did not improve the recorded aggregate metrics over the baseline.

A separate saved example at threshold `0.5` asked “What is the weather in Jakarta?” It retrieved zero passages and returned “I couldn't find relevant information.” No source should be attributed to this fallback because no passage was retrieved.

These observations support threshold filtering as a possible way to reject some unrelated results. They do not establish a universally suitable threshold or an improvement in answer quality.

#### Result (Threshold = 0.5)

Library AI Chatbot!

Menu:
1. Load Data
2. Ask LLM
3. Exit

Question: What is the weather in Jakarta?
No relevant docs were retrieved using the relevance score threshold 0.5

Found 0 documents
I couldn't find relevant information.

Type Quit or quit to back to main menu.
Question: How does Alice get out of Wonderland?

Found 5 documents
Answer: Alice gets out of Wonderland by climbing back through the rabbit hole.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 7
Evidence: Alice was not a bit hurt, and she jumped up on to her feet in a moment:
she looked up, but it was all dark overhead; before her was another
long passage, and the White Rabbit was still in sight, hurrying down
it. There was not a moment to be lost: away went Alice like the wind,
and was just in time to hear it say, as it turned a corner, “Oh my ears
and whiskers, how late it’s getting!” She was close behind it when she
turned the corner, but the Rabbit was no longer to be seen: she found
herself in a long, low hall, which was lit up by a row of lamps hanging
from the roof.

There were doors all round the hall, but they were all locked; and when
Alice had been all the way down one side and up the other, trying every
door, she walked sadly down the middle, wondering how she was ever to
get out again.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 0
Evidence: Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster Quadrille
 CHAPTER XI.    Who Stole the Tarts?
 CHAPTER XII.   Alice’s Evidence

CHAPTER I.
Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 2
Evidence: There was nothing so _very_ remarkable in that; nor did Alice think it
so _very_ much out of the way to hear the Rabbit say to itself, “Oh
dear! Oh dear! I shall be late!” (when she thought it over afterwards,
it occurred to her that she ought to have wondered at this, but at the
time it all seemed quite natural); but when the Rabbit actually _took a
watch out of its waistcoat-pocket_, and looked at it, and then hurried
on, Alice started to her feet, for it flashed across her mind that she
had never before seen a rabbit with either a waistcoat-pocket, or a
watch to take out of it, and burning with curiosity, she ran across the
field after it, and fortunately was just in time to see it pop down a
large rabbit-hole under the hedge.

In another moment down went Alice after it, never once considering how
in the world she was to get out again.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 3
Evidence: The rabbit-hole went straight on like a tunnel for some way, and then
dipped suddenly down, so suddenly that Alice had not a moment to think
about stopping herself before she found herself falling down a very
deep well.

Either the well was very deep, or she fell very slowly, for she had
plenty of time as she went down to look about her and to wonder what
was going to happen next. First, she tried to look down and make out
what she was coming to, but it was too dark to see anything; then she
looked at the sides of the well, and noticed that they were filled with
cupboards and book-shelves; here and there she saw maps and pictures
hung upon pegs. She took down a jar from one of the shelves as she
passed; it was labelled “ORANGE MARMALADE”, but to her great
disappointment it was empty: she did not like to drop the jar for fear
of killing somebody underneath, so managed to put it into one of the
cupboards as she fell past it.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 51
Evidence: “Oh! So Bill’s got to come down the chimney, has he?” said Alice to
herself. “Shy, they seem to put everything upon Bill! I wouldn’t be in
Bill’s place for a good deal: this fireplace is narrow, to be sure; but
I _think_ I can kick a little!”

She drew her foot as far down the chimney as she could, and waited till
she heard a little animal (she couldn’t guess of what sort it was)
scratching and scrambling about in the chimney close above her: then,
saying to herself “This is Bill,” she gave one sharp kick, and waited
to see what would happen next.

The first thing she heard was a general chorus of “There goes Bill!”
then the Rabbit’s voice along—“Catch him, you by the hedge!” then
silence, and then another confusion of voices—“Hold up his head—Brandy
now—Don’t choke him—How was it, old fellow? What happened to you? Tell
us all about it!”

Type Quit or quit to back to main menu.

Goodbye

Menu:
1. Load Data
2. Ask LLM
3. Exit

Thank you for using Library AI Chatbot!

### Optional Exploration: Follow-up Question Rewriting

#### What did I experiment with?
Creating a query-rewriting prompt and adding chat history.
#### Why did I do this experiment?
To help the LLM answer follow-up questions from the user.
#### How did I conduct this experiment?
By creating a prompt that instructs the LLM to rewrite follow-up questions containing references such as “she,” “there,” and “it” into standalone questions.
#### What did I find?
Some questions were successfully rewritten in earlier tests. However, when the user entered another standalone question, the model sometimes rewrote it by connecting it to the previous conversation unnecessarily. In the latest test, it also generated answers instead of rewritten questions, sometimes introducing incorrect information. Therefore, the approach worked in some cases but did not consistently preserve the meaning of the user’s question.

#### Result

Library AI Chatbot!

Menu:
1. Load Data
2. Ask LLM
3. Exit

Question: Who does Alice follow down the rabbit hole?

Found 5 documents
Answer: Alice follows the White Rabbit down the rabbit hole.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 4
Evidence: The rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had not a moment to think about stopping herself before she found herself falling down a very deep well.  Either the well was very deep...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 1
Evidence: CHAPTER I.     Down the Rabbit-Hole  CHAPTER II.    The Pool of Tears  CHAPTER III.   A Caucus-Race and a Long Tale  CHAPTER IV.    The Rabbit Sends in a Little Bill  CHAPTER V.     Advice from a Caterpillar  CHAPTER VI.    Pig and Pepper  CHAPTER VI...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 45
Evidence: “How queer it seems,” Alice said to herself, “to be going messages for a rabbit! I suppose Dinah’ll be sending me on messages next!” And she began fancying the sort of thing that would happen: “‘Miss Alice! Come here directly, and get ready for your ...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 161
Evidence: Alice watched the White Rabbit as he fumbled over the list, feeling very curious to see what the next witness would be like, “—for they haven’t got much evidence _yet_,” she said to herself. Imagine her surprise, when the White Rabbit read out, at th...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 3
Evidence: There was nothing so _very_ remarkable in that; nor did Alice think it so _very_ much out of the way to hear the Rabbit say to itself, “Oh dear! Oh dear! I shall be late!” (when she thought it over afterwards, it occurred to her that she ought to hav...

Type Quit or quit to back to main menu.

Searching for: She follows the White Rabbit because he summons her into his lair in Wonderland.
Question: Why does she follow him?

Found 5 documents
Answer: She follows the White Rabbit because he summons her into his lair in Wonderland.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 44
Evidence: Very soon the Rabbit noticed Alice, as she went hunting about, and called out to her in an angry tone, “Why, Mary Ann, what _are_ you doing out here? Run home this moment, and fetch me a pair of gloves and a fan! Quick, now!” And Alice was so much fr...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 8
Evidence: Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the White Rabbit was still in sight, hurrying down it. There was not a moment to be lost: aw...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 20
Evidence: After a time she heard a little pattering of feet in the distance, and she hastily dried her eyes to see what was coming. It was the White Rabbit returning, splendidly dressed, with a pair of white kid gloves in one hand and a large fan in the other:...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 45
Evidence: “How queer it seems,” Alice said to herself, “to be going messages for a rabbit! I suppose Dinah’ll be sending me on messages next!” And she began fancying the sort of thing that would happen: “‘Miss Alice! Come here directly, and get ready for your ...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 43
Evidence: It was the White Rabbit, trotting slowly back again, and looking anxiously about as it went, as if it had lost something; and she heard it muttering to itself “The Duchess! The Duchess! Oh my dear paws! Oh my fur and whiskers! She’ll get me executed,...

Type Quit or quit to back to main menu.

Searching for: Yes, Alice eventually gets out of Wonderland when she reaches the end of the rabbit hole and emerges back into reality.
Question: Does she eventually get out of Wonderland?

Found 5 documents
Answer: Yes, Alice eventually gets out of Wonderland when she reaches the end of the rabbit hole and emerges back into reality.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 8
Evidence: Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the White Rabbit was still in sight, hurrying down it. There was not a moment to be lost: aw...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 4
Evidence: The rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had not a moment to think about stopping herself before she found herself falling down a very deep well.  Either the well was very deep...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 3
Evidence: There was nothing so _very_ remarkable in that; nor did Alice think it so _very_ much out of the way to hear the Rabbit say to itself, “Oh dear! Oh dear! I shall be late!” (when she thought it over afterwards, it occurred to her that she ought to hav...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 175
Evidence: Lastly, she pictured to herself how this same little sister of hers would, in the after-time, be herself a grown woman; and how she would keep, through all her riper years, the simple and loving heart of her childhood: and how she would gather about ...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 89
Evidence: “I should like it very much,” said Alice, “but I haven’t been invited yet.”  “You’ll see me there,” said the Cat, and vanished.  Alice was not much surprised at this, she was getting so used to queer things happening. While she was looking at the pla...

Type Quit or quit to back to main menu.

Searching for: Alice gets out of Wonderland by reaching the end of the rabbit hole and emerging back into reality.
Question: How does she get out?

Found 5 documents
Answer: Alice gets out of Wonderland by reaching the end of the rabbit hole and emerging back into reality.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 8
Evidence: Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the White Rabbit was still in sight, hurrying down it. There was not a moment to be lost: aw...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 4
Evidence: The rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had not a moment to think about stopping herself before she found herself falling down a very deep well.  Either the well was very deep...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 3
Evidence: There was nothing so _very_ remarkable in that; nor did Alice think it so _very_ much out of the way to hear the Rabbit say to itself, “Oh dear! Oh dear! I shall be late!” (when she thought it over afterwards, it occurred to her that she ought to hav...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 175
Evidence: Lastly, she pictured to herself how this same little sister of hers would, in the after-time, be herself a grown woman; and how she would keep, through all her riper years, the simple and loving heart of her childhood: and how she would gather about ...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 73
Evidence: “Well, be off, then!” said the Pigeon in a sulky tone, as it settled down again into its nest. Alice crouched down among the trees as well as she could, for her neck kept getting entangled among the branches, and every now and then she had to stop an...

Type Quit or quit to back to main menu.

Searching for: Sherlock Holmes is a fictional detective created by British author Sir Arthur Conan Doyle. He is known for solving complex cases using logical reasoning and deductive skills.
Question: Who is Sherlock Holmes?

Found 5 documents
Answer: Sherlock Holmes is a fictional detective created by British author Sir Arthur Conan Doyle. He is known for solving complex cases using logical reasoning and deductive skills.

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 249
Evidence: When I glance over my notes and records of the Sherlock Holmes cases between the years ’82 and ’90, I am faced by so many which present strange and interesting features that it is no easy matter to know which to choose and which to leave. Some, howev...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 251
Evidence: The year ’87 furnished us with a long series of cases of greater or less interest, of which I retain the records. Among my headings under this one twelve months I find an account of the adventure of the Paradol Chamber, of the Amateur Mendicant Socie...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 422
Evidence: On glancing over my notes of the seventy odd cases in which I have during the last eight years studied the methods of my friend Sherlock Holmes, I find many tragic, some comic, a large number merely strange, but none commonplace; for, working as he d...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 282
Evidence: Sherlock Holmes closed his eyes and placed his elbows upon the arms of his chair, with his finger-tips together. “The ideal reasoner,” he remarked, “would, when he had once been shown a single fact in all its bearings, deduce from it not only all the...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 349
Evidence: A large and comfortable double-bedded room had been placed at our disposal, and I was quickly between the sheets, for I was weary after my night of adventure. Sherlock Holmes was a man, however, who, when he had an unsolved problem upon his mind, wou...

Type Quit or quit to back to main menu.

Searching for: His friend and companion is Dr. John Watson.
Question: Who is his friend and companion?

Found 5 documents
Answer: Dr. John Watson is Holmes' friend and companion mentioned in the passage.

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 143
Evidence: “You have made your position very clear to me,” said Holmes. “This is my friend, Dr. Watson, before whom you can speak as freely as before myself. Kindly tell us now all about your connection with Mr. Hosmer Angel.”

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 64
Evidence: I had called upon my friend, Mr. Sherlock Holmes, one day in the  autumn of last year and found him in deep conversation with a very  stout, florid-faced, elderly gentleman with fiery red hair. With an  apology for my intrusion, I was about to withdr...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 66
Evidence: “You did, Doctor, but none the less you must come round to my view, for otherwise I shall keep on piling fact upon fact on you until your reason breaks down under them and acknowledges me to be right. Now, Mr. Jabez Wilson here has been good enough t...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 19
Evidence: “You had my note?” he asked with a deep harsh voice and a strongly marked German accent. “I told you that I would call.” He looked from one to the other of us, as if uncertain which to address.  “Pray take a seat,” said Holmes. “This is my friend and...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 457
Evidence: “Fancy his having the insolence to confound me with the official detective force! This incident gives zest to our investigation, however, and I only trust that our little friend will not suffer from her imprudence in allowing this brute to trace her....

Type Quit or quit to back to main menu.

Searching for: Alice and the Caterpillar talk about tea and flowers.
Question: What do Alice and the Caterpillar talk about?

Found 5 documents
Answer: Alice and the Caterpillar talk about tea and flowers.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 61
Evidence: This was not an encouraging opening for a conversation. Alice replied, rather shyly, “I—I hardly know, sir, just at present—at least I know who I _was_ when I got up this morning, but I think I must have been changed several times since then.”  “What...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 62
Evidence: “Well, perhaps your feelings may be different,” said Alice; “all I know is, it would feel very queer to _me_.”  “You!” said the Caterpillar contemptuously. “Who are _you?_”  Which brought them back again to the beginning of the conversation. Alice fe...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 65
Evidence: “You are old,” said the youth, “one would hardly suppose     That your eye was as steady as ever; Yet you balanced an eel on the end of your nose—     What made you so awfully clever?”  “I have answered three questions, and that is enough,”     Said ...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 66
Evidence: “Are you content now?” said the Caterpillar.  “Well, I should like to be a _little_ larger, sir, if you wouldn’t mind,” said Alice: “three inches is such a wretched height to be.”  “It is a very good height indeed!” said the Caterpillar angrily, rear...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 60
Evidence: The great question certainly was, what? Alice looked all round her at the flowers and the blades of grass, but she did not see anything that looked like the right thing to eat or drink under the circumstances. There was a large mushroom growing near ...

Type Quit or quit to back to main menu.

Searching for: The weather in Jakarta can vary depending on the time of year, but generally, it's hot and humid with temperatures ranging from 25°C to 30°C (77°F to 86°F) during the day and around 19°C to 24°C (66°F to 75°F) at night. It also experiences heavy rainfall throughout the year, especially during the monsoon season.
Question: What is the weather in Jakarta?

Found 5 documents
Answer: I do not know the answer to the question "The weather in Jakarta can vary depending on the time of year, but generally, it's hot and humid with temperatures ranging from 25°C to 30°C (77°F to 86°F) during the day and around 19°C to 24°C (66°F to 75°F) at night. It also experiences heavy rainfall throughout the year, especially during the monsoon season."

Source: Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley –– Chunk 296
Evidence: “The old man, leaning on his son, walked each day at noon, when it did not rain, as I found it was called when the heavens poured forth its waters. This frequently took place, but a high wind quickly dried the earth, and the season became far more pl...

Source: Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley –– Chunk 24
Evidence: So strange an accident has happened to us that I cannot forbear recording it, although it is very probable that you will see me before these papers can come into your possession.  Last Monday (July 31st) we were nearly surrounded by ice, which closed...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 252
Evidence: It was in the latter days of September, and the equinoctial gales had set in with exceptional violence. All day the wind had screamed and the rain had beaten against the windows, so that even here in the heart of great, hand-made London we were force...

Source: Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley –– Chunk 267
Evidence: “One day, when I was oppressed by cold, I found a fire which had been left by some wandering beggars, and was overcome with delight at the warmth I experienced from it. In my joy I thrust my hand into the live embers, but quickly drew it out again wi...

Source: Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley –– Chunk 537
Evidence: a great and sudden change. The sun might shine or the clouds might lower, but nothing could appear to me as it had done the day before. A fiend had snatched from me every hope of future happiness; no creature had ever been so miserable as I was; so f...

Type Quit or quit to back to main menu.

Goodbye

Menu:
1. Load Data
2. Ask LLM
3. Exit

Thank you for using Library AI Chatbot!

## Limitations and Uncertainties

### Known Limitations

- **Retrieval:** Similarity can match the right character but the wrong event. Top-k retrieval can return irrelevant passages.
- **Generation:** Qwen can omit the requested fact or make unsupported claims even when relevant evidence is available.
- **Attribution:** Displayed references list retrieved passages, they do not prove that each answer claim is supported.
- **Chunking and cleaning:** Zero overlap can separate related information. Contents pages and some illustration text remain.
- **Evaluation:** Five selected questions cover only Alice and Sherlock Holmes, not the complete collection. The same small model generates and evaluates answers.

### Questions That Remain Open

| Uncertainty or assumption | How I would investigate it |
|---|---|
| Chunk overlap might improve retrieval, but I have not tested it yet. | Compare overlap values of `0`, `100`, and `200`, keeping other settings the same. Update reference chunk IDs after rebuilding. |
| BGE has not been compared with other embedding models on these books. | Test other embedding models using the same questions and chunking. Compare Hit@5, MRR@5, and processing time. |
|Retrieving five chunks may not be the most suitable setting, as other values of k have not been tested. |Compare k=3, 5, and 10. Check retrieval results, answer quality, and response time.|
| Providing the correct evidence may improve answer quality, but the model may still make mistakes.	| Compare answers using retrieved passages and manually selected reference passages. Keep the question, model, and prompt the same. |
| Some wrong answers may come from the generation model rather than retrieval. | Compare different models using the same questions and retrieved passages. Check answer quality and response time. |

These are open questions or assumptions, not observed improvements.

## Proposed Future Work


The following work is **not completed**. It is listed in the order I would prioritize it.


1. **Investigate retrieval failures.** Test chunk overlap, other embedding models, or retrieval methods to see whether they find more relevant passages.
2. **Investigate generation and evaluator failures separately.** Test changes to the prompt or model using the same questions and retrieved passages. Separately, check whether the automatic evaluation scores agree with manual reviews.
3. **Expand and stabilize evaluation.** Add more questions across all four books, including questions requiring multiple passages and questions the books cannot answer. Save the settings and results for comparison.
4. **Revisit follow-up questions.** Test whether question rewriting handles references such as “she” and “it” correctly while leaving unrelated questions unchanged.
5. **Improve ingestion reliability.** Make sure documents are stored completely without duplicates, including when loading is interrupted or files are updated.

For each experiment, I would keep unrelated settings fixed, report both successes and failures, and compare against the saved baseline. I would not claim an improvement until the results support it.